# 🏏 Cricket Score Predictor System

Predict the **final innings score** from the current match situation.

**Steps:** load data → explore → feature engineering → train → evaluate → save → predict

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load data
Uses `data/cricket_data.csv` if it exists, otherwise generates sample data so the notebook runs end to end.

Required columns: `current_score, overs, wickets, last_5_overs, final_score`

In [ ]:
DATA_PATH = Path('data/cricket_data.csv')
rng = np.random.default_rng(42)

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print('Loaded real dataset:', df.shape)
else:
    n = 4000
    overs = rng.uniform(5, 45, n).round(1)
    run_rate = rng.normal(5.6, 0.9, n).clip(3, 9)
    current_score = (overs * run_rate).round()
    wickets = np.clip((overs * 0.18 + rng.normal(0, 1.5, n)).round(), 0, 9)
    last_5_overs = (5 * run_rate + rng.normal(0, 4, n)).clip(10, 70).round()
    remaining = 50 - overs
    future_rate = (0.6 * run_rate + 0.4 * last_5_overs / 5) * (1 - 0.04 * wickets) + 0.4
    final_score = (current_score + remaining * future_rate + rng.normal(0, 8, n)).round()
    df = pd.DataFrame({
        'current_score': current_score,
        'overs': overs,
        'wickets': wickets,
        'last_5_overs': last_5_overs,
        'final_score': final_score,
    })
    print('Real dataset not found. Using generated sample data:', df.shape)

df.head()

## 2. Explore the data

In [ ]:
print(df.info())
print(df.isnull().sum())
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['final_score'], bins=30)
axes[0].set_title('Distribution of final scores')
axes[0].set_xlabel('Final score')

axes[1].scatter(df['current_score'], df['final_score'], s=5, alpha=0.4)
axes[1].set_title('Current score vs final score')
axes[1].set_xlabel('Current score')
axes[1].set_ylabel('Final score')
plt.tight_layout()
plt.show()

## 3. Clean data and engineer features

In [ ]:
df = df.dropna().drop_duplicates()
df = df[df['overs'] > 0]

df['run_rate'] = df['current_score'] / df['overs']
df['balls_left'] = (50 - df['overs']) * 6   # change 50 to 20 for T20 data

FEATURES = ['current_score', 'overs', 'wickets', 'last_5_overs', 'run_rate', 'balls_left']
TARGET = 'final_score'

X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 4. Train models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print('Trained:', name)

## 5. Evaluate

In [ ]:
rows = []
for name, model in models.items():
    pred = model.predict(X_test)
    rows.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred),
    })

results = pd.DataFrame(rows).round(3)
results

In [ ]:
best_name = results.sort_values('MAE').iloc[0]['Model']
best_model = models[best_name]
pred = best_model.predict(X_test)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, s=5, alpha=0.4)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual final score')
plt.ylabel('Predicted final score')
plt.title('Best model: ' + best_name)
plt.show()

## 6. Save the best model

In [ ]:
Path('models').mkdir(exist_ok=True)
joblib.dump(best_model, 'models/score_predictor.pkl')
print('Saved models/score_predictor.pkl')

## 7. Predict a live score

In [ ]:
def predict_score(current_score, overs, wickets, last_5_overs, total_overs=50):
    model = joblib.load('models/score_predictor.pkl')
    row = pd.DataFrame([{
        'current_score': current_score,
        'overs': overs,
        'wickets': wickets,
        'last_5_overs': last_5_overs,
        'run_rate': current_score / overs,
        'balls_left': (total_overs - overs) * 6,
    }])
    return round(float(model.predict(row[FEATURES])[0]))

print('Predicted final score:', predict_score(120, 20, 3, 48))